## Ano

In [11]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.covariance import LedoitWolf
import seaborn as sns
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
from tqdm import tqdm
import pickle
from functions import (
    corr_matrix, get_network_TMFG, build_and_save_graph
)

years = [2019, 2020, 2021, 2022, 2023, 2024]

modes = {}

for year in years:
    modes[f"fast_{year}"] = pd.read_parquet(f"../../data/03_modes/{year}/fast_emd.parquet")
    modes[f"medium_{year}"] = pd.read_parquet(f"../../data/03_modes/{year}/medium_emd.parquet")
    modes[f"slow_{year}"] = pd.read_parquet(f"../../data/03_modes/{year}/slow_emd.parquet")

In [12]:
corr = {}

for year in years:
    corr[f"corr_fast_{year}"] = corr_matrix(modes[f"fast_{year}"])
    corr[f"corr_medium_{year}"] = corr_matrix(modes[f"medium_{year}"])
    corr[f"corr_slow_{year}"] = corr_matrix(modes[f"slow_{year}"])

In [15]:
graph = {}

# Lista de todas as tarefas (ano, velocidade)
tasks = [(year, speed) for year in years for speed in ["fast", "medium", "slow"]]

# Barra de progresso
for year, speed in tqdm(tasks, desc="Construindo grafos"):
    # Recupera a matriz de correlação (DataFrame com nomes dos ativos)
    corr_matrix = corr[f"corr_{speed}_{year}"]

    # Constroi o grafo TMFG
    G = get_network_TMFG(corr_matrix)

    # Cria a pasta se não existir e salva
    base_path = f"../../data/04_graphs/{year}"
    os.makedirs(base_path, exist_ok=True)
    file_path = os.path.join(base_path, f"{speed}_graph.gpickle")
    with open(file_path, "wb") as f:
        pickle.dump(G, f)

    # Armazena no dicionário
    graph[f"graph_{speed}_{year}"] = G

Construindo grafos: 100%|██████████| 18/18 [51:15<00:00, 170.88s/it]


## Ano-mês

In [10]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.covariance import LedoitWolf
import seaborn as sns
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
from tqdm import tqdm
import pickle
from functions import (
    corr_matrix, get_network_TMFG, build_and_save_graph
)

base_modes = "../../data/03_modes"

modes = {}

# List all year_month folders
year_months = sorted([
    d for d in os.listdir(base_modes)
    if "_" in d and os.path.isdir(os.path.join(base_modes, d))
])

for ym in year_months:
    year, month = ym.split("_")

    modes[f"fast_{ym}"] = pd.read_parquet(
        f"{base_modes}/{ym}/fast_emd.parquet"
    )
    modes[f"medium_{ym}"] = pd.read_parquet(
        f"{base_modes}/{ym}/medium_emd.parquet"
    )
    modes[f"slow_{ym}"] = pd.read_parquet(
        f"{base_modes}/{ym}/slow_emd.parquet"
    )

corr = {}
skipped = []

for ym in year_months:
    for speed in ["fast", "medium", "slow"]:
        key = f"{speed}_{ym}"

        try:
            corr[f"corr_{speed}_{ym}"] = corr_matrix(
                modes[key],
                min_obs=10
            )
        except ValueError as e:
            skipped.append((ym, speed, str(e)))

In [12]:
import os
import pickle
from tqdm import tqdm

graph = {}

# Detectar todos os year_month disponíveis a partir das correlações
year_months = sorted({
    key.replace("corr_fast_", "")
    for key in corr.keys()
    if key.startswith("corr_fast_")
})

# Lista de tarefas (year_month, speed)
tasks = [(ym, speed) for ym in year_months for speed in ["fast", "medium", "slow"]]

# Barra de progresso
for ym, speed in tqdm(tasks, desc="Construindo grafos (ano-mês)"):

    # Recupera a matriz de correlação
    corr_matrix_ = corr[f"corr_{speed}_{ym}"]

    # Constrói o grafo TMFG
    G = get_network_TMFG(corr_matrix_)

    # Cria a pasta e salva
    base_path = f"../../data/04_graphs/{ym}"
    os.makedirs(base_path, exist_ok=True)

    file_path = os.path.join(base_path, f"{speed}_graph.gpickle")
    with open(file_path, "wb") as f:
        pickle.dump(G, f)

    # Armazena no dicionário
    graph[f"graph_{speed}_{ym}"] = G

Construindo grafos (ano-mês): 100%|██████████| 216/216 [9:07:35<00:00, 152.11s/it]  
